# Momentum ranker — Nasdaq-100

Runs the **same code the live trader runs** — `cross_sectional_momentum` and
`book_vol_target` from the repo — and shows today's ranking, today's target book,
and how the held names' ranks have moved.

Nothing here reimplements the strategy. If a number differs from the VM, the
cause is the data or a parameter, never a second copy of the logic.

**To use:** Runtime → Run all. Takes about a minute, most of it the download.

## 1 · Setup

In [ ]:
# The repo is public, so no token is needed.
!git clone --depth 1 https://github.com/tli9181991/qqq_boxx_strategies.git 2>/dev/null || git -C qqq_boxx_strategies pull --ff-only
!pip install -q "yfinance>=0.2.40" "lxml>=4.9"

import os, sys
REPO = "/content/qqq_boxx_strategies"
os.chdir(REPO)                 # the price cache lives at data/, resolved relatively
sys.path.insert(0, REPO)

import pandas as pd
pd.set_option("display.width", 160)
pd.set_option("display.max_rows", 120)
print("repo at", REPO, "@", os.popen("git rev-parse --short HEAD").read().strip())

## 2 · Parameters

These mirror `qbs/config.py`. Change them here to explore; the deployed book
uses the defaults, and changing them in this notebook changes nothing on the VM.

In [ ]:
from dataclasses import replace
from qbs.config import Config

cfg = Config()

N_HOLD      = cfg.momentum.n_hold        # 6   names held
EXIT_RANK   = cfg.momentum.exit_rank     # 10  sell once rank passes this
SHOW_TOP    = 25                         # how much of the ranking to print
TARGET_VOL  = cfg.book_vol.target_vol    # 0.25 annualised, the book-level target

cfg.momentum = replace(cfg.momentum, n_hold=N_HOLD, exit_rank=EXIT_RANK)
cfg.book_vol = replace(cfg.book_vol, target_vol=TARGET_VOL)

print(f"top {N_HOLD}, exit past rank {EXIT_RANK}, "
      f"12-{cfg.momentum.skip_months} momentum, target vol {TARGET_VOL:.0%}")
print(f"absolute filter: {cfg.momentum.absolute_filter} "
      f"(a name must also beat {cfg.momentum.safe_asset})")

## 3 · Data

Downloads the Nasdaq-100 constituents plus the safe asset, by the same two paths
`pipeline.run()` uses — the universe from its batched cache, the safe asset on its
own. They are separate because the universe cache holds index members only, and
asking it for BOXX would silently drop the cash leg.

In [ ]:
from qbs.universe import load_universe
from qbs.live.signals import load_live_prices

tickers = load_universe(fetch=True, warn=False)
px = load_live_prices(cfg, tickers=tickers, refresh=True)

print(f"{len(tickers)} tickers · {px.shape[0]} rows × {px.shape[1]} cols")
print(f"last bar {px.index.max():%Y-%m-%d}")

## 4 · Today's ranking and book

`compute_targets` is the live signal path: it runs the strategy over the full
history and returns the last row. The `held` column shows the hysteresis band at
work — a name ranked 7-10 is kept, but a name at 7 that was never bought is not.

In [ ]:
from qbs.live.signals import compute_targets

book = compute_targets(cfg, px, requested=tickers, record_ranks=SHOW_TOP,
                       max_staleness_days=5, min_coverage=0.85)

print(book.describe())
print()

rank = pd.DataFrame(book.ranking)
rank["score"] = rank["score"].map(lambda v: f"{v:+.1%}")
rank["held"] = rank["held"].map({True: "HELD", False: ""})
rank["zone"] = [("book" if r <= N_HOLD else "band" if r <= EXIT_RANK else "queue")
                for r in rank["rank"]]
display(rank[["rank", "symbol", "score", "held", "zone"]].set_index("rank"))

## 5 · Rank trajectories

Rank 1 is at the top. The two guides are the decision boundaries: a name is bought
when it reaches the top **N_HOLD**, and sold once it falls past **EXIT_RANK**. A
line drifting toward the lower guide is a rotation about to happen.

In [ ]:
import matplotlib.pyplot as plt
from qbs.strategies import cross_sectional_momentum

LOOKBACK = 120     # sessions of history to draw

mom = cross_sectional_momentum(
    px.drop(columns=[cfg.momentum.safe_asset]), px[cfg.momentum.safe_asset],
    cfg.momentum, record_ranks=60)

hist = (pd.DataFrame([{"date": d, "symbol": t, "rank": r}
                      for d, rows in (mom.rank_log or {}).items()
                      for t, r, _ in rows])
        .pivot(index="date", columns="symbol", values="rank")
        .tail(LOOKBACK))

held = [t for t in book.raw_holdings if t in hist.columns]
# Fixed categorical order, never cycled -- slot i always means the same name here.
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300",
           "#4a3aa7", "#e34948"]

fig, ax = plt.subplots(figsize=(11, 5.5))
fig.patch.set_facecolor("#fcfcfb"); ax.set_facecolor("#fcfcfb")

for i, t in enumerate(held[:len(PALETTE)]):
    s = hist[t].dropna()
    ax.plot(s.index, s.values, lw=2, color=PALETTE[i], solid_capstyle="round")
    ax.plot(s.index[-1], s.values[-1], "o", ms=8, color=PALETTE[i],
            mec="#fcfcfb", mew=2)
    # Direct labels: identity is never carried by colour alone.
    ax.annotate(f" {t}", (s.index[-1], s.values[-1]), color=PALETTE[i],
                va="center", fontsize=10, fontweight="600")

# Room at the right for the direct labels, which are the identity encoding.
span = hist.index[-1] - hist.index[0]
ax.set_xlim(hist.index[0], hist.index[-1] + span * 0.07)
ax.axhline(N_HOLD, color="#8a8a80", lw=1, ls="--")
ax.axhline(EXIT_RANK, color="#8a8a80", lw=1, ls="--")
ax.annotate(f"bought at {N_HOLD}", (hist.index[0], N_HOLD), color="#6b6b62",
            fontsize=9, va="bottom")
ax.annotate(f"sold past {EXIT_RANK}", (hist.index[0], EXIT_RANK), color="#6b6b62",
            fontsize=9, va="bottom")

ax.invert_yaxis()
# Clipped to the decision zone. A name that was once rank 40 is not the
# question; how close it is to the exit boundary is, and an axis stretched to
# fit its history squashes every rank that matters into a few pixels.
FLOOR = EXIT_RANK + 8
ax.set_ylim(FLOOR, 0.4)
ax.set_yticks([1, N_HOLD, EXIT_RANK, FLOOR])
ax.set_ylabel("rank  (1 = strongest)", color="#3d3d38")
ax.set_title(f"12-1 momentum rank, last {LOOKBACK} sessions",
             color="#1a1a19", fontsize=13, fontweight="600", loc="left")
ax.grid(axis="y", color="#e5e5df", lw=0.8)
ax.set_axisbelow(True)
for side in ("top", "right", "left"):
    ax.spines[side].set_visible(False)
ax.spines["bottom"].set_color("#d5d5cd")
ax.tick_params(colors="#6b6b62", length=0)
plt.tight_layout(); plt.show()

display(hist[held].tail(10).astype("Int64"))

## 6 · What would trade

Target share counts for a given book size, using the same `floor()` sizing and the
same no-trade band as the live runner. Set `HELD_NOW` to what you actually hold to
see the order list.

In [ ]:
from qbs.live.orders import build_orders, format_order_table

NOTIONAL = 100_000
HELD_NOW = {}          # e.g. {"MU": 6, "AMD": 11, "BOXX": 539}; {} = a flat book

orders, target = build_orders(
    book.weights, book.prices, HELD_NOW,
    notional=NOTIONAL,
    max_order_notional=40_000, max_gross_turnover=1.60, max_positions=12,
    min_notional=250.0, min_drift=0.25,
    safe_asset=cfg.momentum.safe_asset, universe=book.universe)

print(format_order_table(orders, target, HELD_NOW))

---

**Caveats worth keeping in mind**

* The ranking is as of the **last complete bar**. Run this intraday and the last
  bar is a partial day — which is deliberate in the live trader (it decides at
  ~15:30 and fills at the close, the gap being what `slippage_bps` models), but
  means a midday ranking here is not the ranking the 15:30 job will see.
* The universe is **today's** Nasdaq-100 membership, so historical ranks carry
  survivorship bias. `qbs/universe.py` has point-in-time membership if you need
  the unbiased version.
* Changing the parameters in section 2 changes nothing on the VM. `exit_rank` and
  `n_hold` live in `qbs/config.py` and reach the live book only through a commit.